# Estudo: Tratamento de Dados Desbalanceados

<p align="left">
  <img src="img_01.png" width="800">
</p>

In [1]:
# ============================================================
# GERANDO UMA MASSA DE DADOS DESBALANCEADA
# Objetivo:
# Simular um dataset para estudos de:
# - Oversampling
# - Undersampling
# - SMOTE
# - Classificação desbalanceada
# ============================================================
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

 
#  CONFIGURAÇÃO DO DATASET
 

In [2]:
X, y = make_classification(
    n_samples=10000,          # quantidade total de registros
    n_features=10,            # número total de variáveis
    n_informative=6,          # variáveis realmente importantes
    n_redundant=2,            # variáveis redundantes
    n_repeated=0,
    n_classes=2,              # classificação binária
    weights=[0.70, 0.30],     # DESBALANCEAMENTO
    flip_y=0.01,              # pequeno ruído
    random_state=42
)

# TRANSFORMANDO EM DATAFRAME


In [3]:
colunas = [f'feature_{i}' for i in range(1, 11)]
dataset = pd.DataFrame(X, columns=colunas)
# variável alvo
dataset['inadimplente'] = y

# VISUALIZAÇÃO INICIAL

In [4]:
dataset.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,inadimplente
0,-3.079755,-1.023890,-2.059742,-0.852741,0.227333,0.493899,0.404851,0.627201,0.646710,-1.111949,1
1,0.028729,0.972903,-0.302152,0.408143,0.283663,0.109245,-0.356853,-1.255259,-1.953986,-2.798042,0
2,1.402790,2.192526,0.658497,0.631485,1.106866,-0.843229,0.188334,-1.145284,-2.836721,-3.579858,1
3,0.895957,0.663586,0.549948,-1.324490,0.714363,2.226197,-1.680641,-0.157837,-4.041746,-1.662902,0
4,1.532724,3.085354,-1.899169,-1.144469,-1.105577,3.233514,-2.509374,-1.055282,-2.406049,0.488451,0


In [9]:
print('\nDistribuição da variável alvo:\n')
print(dataset['inadimplente'].value_counts().reset_index())


Distribuição da variável alvo:

   inadimplente  count
0             0   6979
1             1   3021


In [8]:
print('\nDistribuição percentual:\n')
print(dataset['inadimplente'].value_counts(normalize=True) * 100)


Distribuição percentual:

inadimplente
0    69.79
1    30.21
Name: proportion, dtype: float64


In [10]:
# Esta função calcula a prevalência da classe positiva (label = 1)
def calcula_prevalencia(y_actual):
    return sum(y_actual) / len(y_actual)  # Calcula a prevalência dividindo a soma dos valores em y_atual pelo número total de elementos em y_atual

In [11]:
# Imprime a prevalência da classe positiva no DataFrame df_dsa["LABEL_TARGET"].values com três casas decimais
print("Prevalência da classe positiva: %.3f" % calcula_prevalencia(dataset["inadimplente"].values)) 

Prevalência da classe positiva: 0.302


# Divisão dos Dados Mantendo a Prevalência de Classe


In [13]:
dados_aleatorios = dataset.sample(n=len(dataset))
dados_aleatorios.head(5)

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,inadimplente
2118,-2.003369,-2.351223,-0.127954,-1.115054,-0.056555,0.098522,-0.767327,0.421188,-0.300453,-1.441551,0
3421,-0.038332,-0.430074,-0.001036,2.102868,-1.000635,-1.199162,0.314158,2.223327,1.688942,-0.116850,1
1274,-0.248162,-1.817813,1.165548,-1.762178,-0.656204,-1.070481,-1.420489,0.079067,-0.784538,-1.596643,0
5387,-1.944511,1.002714,-0.932358,-0.122263,0.131126,2.119584,1.350070,1.067051,-4.357840,-4.256528,0
1571,-2.546436,-1.574876,-0.202707,-3.229172,-1.084949,-1.931711,0.653267,0.132569,-0.124893,-2.577649,0


In [14]:
dados_aleatorios = dados_aleatorios.reset_index(drop=True)
dados_aleatorios

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,inadimplente
0,-2.003369,-2.351223,-0.127954,-1.115054,-0.056555,0.098522,-0.767327,0.421188,-0.300453,-1.441551,0
1,-0.038332,-0.430074,-0.001036,2.102868,-1.000635,-1.199162,0.314158,2.223327,1.688942,-0.116850,1
2,-0.248162,-1.817813,1.165548,-1.762178,-0.656204,-1.070481,-1.420489,0.079067,-0.784538,-1.596643,0
3,-1.944511,1.002714,-0.932358,-0.122263,0.131126,2.119584,1.350070,1.067051,-4.357840,-4.256528,0
4,-2.546436,-1.574876,-0.202707,-3.229172,-1.084949,-1.931711,0.653267,0.132569,-0.124893,-2.577649,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,-0.635194,-2.300635,1.060208,1.102730,0.493870,-0.383803,-1.362905,-0.694074,-1.121906,-2.774987,0
9996,0.729949,2.023223,-1.631578,0.245586,-0.065608,1.943328,-1.489215,1.112534,-1.017593,0.068170,0
9997,1.724745,0.451783,1.340993,1.581539,0.680477,-2.027090,0.394268,0.984161,0.979650,0.078989,1
9998,0.895515,-0.741837,0.005159,1.620327,-0.643641,-0.397337,-2.472114,1.341442,0.988338,-0.211274,0


In [15]:
dados_aleatorios_amostra_30 = dados_aleatorios.sample(frac=0.3)
dados_aleatorios_amostra_30

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,inadimplente
1871,1.991610,0.735743,2.726258,1.667152,0.749946,-0.007879,2.169902,0.877266,-2.006434,0.103498,1
7704,-3.716355,-0.401322,-3.488734,-0.690592,0.088081,1.533623,0.105632,1.127135,0.723068,-1.072918,1
9549,-3.107674,-1.098633,-1.855363,-1.515739,0.382456,-0.458691,0.433481,-0.896321,0.807143,-1.652167,0
1288,-2.237805,1.693062,-2.110638,2.973269,-1.205583,0.443281,3.188358,-0.084009,-0.417889,-2.703432,0
2284,0.641870,-0.035773,0.405283,0.304346,0.383775,-0.563701,-0.777119,-0.390556,-0.123043,-0.614199,1
...,...,...,...,...,...,...,...,...,...,...,...
9483,-0.342239,-1.128693,0.737830,1.051378,1.075329,0.028967,-0.842101,0.670347,-1.723526,-2.804585,0
4300,3.421905,1.964456,1.980198,1.488925,-1.812958,-1.991019,0.457261,0.930787,0.117496,0.525282,1
1454,-0.144439,1.190361,-0.832766,1.539019,1.391614,-0.448674,0.105775,0.677821,-0.611088,-2.455768,0
7809,-0.022146,1.029108,-0.382448,1.066490,0.750040,-1.499878,0.974580,-2.483018,0.546646,-1.407630,1


In [16]:
# Calcula e imprime a proporção entre o tamanho de dados_aleatorios_amostra_30 e dados_aleatorios, representando a divisão de validação/teste
print("Tamanho da divisão de validação / teste: %.1f" % (len(dados_aleatorios_amostra_30) / len(dataset)))

Tamanho da divisão de validação / teste: 0.3


## Separação do dataset em treino, teste e validação

In [ ]:
# Cria o conjunto de TESTE selecionando aleatoriamente
# 50% dos registros do dataframe original.

# sample(frac=0.5):
# - frac=0.5 indica que queremos 50% das linhas
# - a seleção ocorre de forma aleatória
# - o resultado é armazenado em "dados_teste"
dados_teste = dados_aleatorios_amostra_30.sample(frac=0.5)

# Cria o conjunto de VALIDAÇÃO removendo do dataframe original
# todas as linhas que já foram selecionadas para o conjunto de teste.
# drop(dados_teste.index):
# - remove os índices presentes em "dados_teste"
# - garante que não existam registros duplicados
#   entre teste e validação
dados_validacao = dados_aleatorios_amostra_30.drop(dados_teste.index)

# Cria o conjunto de TREINO.
# Neste ponto existe um erro de sintaxe/lógica:
# "dados_aleatorios(...)" aparenta ser uma função,
# porém provavelmente o objetivo era selecionar os dados
# restantes para treino.
# O correto dependerá da estratégia desejada.
dados_treino = dataset.drop(dados_aleatorios_amostra_30.index)

In [18]:
# Verifica a prevalência de cada subconjunto e imprime os resultados
print(
    "Teste(n = %d): %.3f"
    % (len(dados_teste), calcula_prevalencia(dados_teste.inadimplente.values))
)
print(
    "Validação(n = %d): %.3f"
    % (len(dados_validacao), calcula_prevalencia(dados_validacao.inadimplente.values))
)
print(
    "Treino(n = %d): %.3f"
    % (len(dados_treino), calcula_prevalencia(dados_treino.inadimplente.values))
)

Teste(n = 1500): 0.309
Validação(n = 1500): 0.295
Treino(n = 7000): 0.304


# Balanceamento de Classe

In [19]:
dados_teste.shape, dados_treino.shape, dados_validacao.shape

((1500, 11), (7000, 11), (1500, 11))

In [20]:
dados_treino['inadimplente'].value_counts()

inadimplente
0    4869
1    2131
Name: count, dtype: int64

In [21]:
# Cria uma variável booleana 'indice' que indica True para linhas onde TARGET é igual a 1 no DataFrame df_treino
indice = dados_treino['inadimplente']== 1

In [22]:
# Define valores positivos e negativos do índice
dados_treino_positivo = dados_treino.loc[indice]    # Cria um DataFrame df_train_pos contendo apenas as linhas onde 'indice' é True, ou seja, onde a TARGET é igual a 1 em dados_treino
dados_treino_negativo = dados_treino.loc[~indice]   # Cria um DataFrame df_train_neg contendo apenas as linhas onde 'indice' é False, ou seja, onde a TARGET não é igual a 1 em dados_treino

In [23]:
# Calcula o valor mínimo entre o número de linhas em dados_treino_positivo e dados_treino_negativo usando a função np.min() do numpy
valor_minimo = np.min([len(dados_treino_positivo), len(dados_treino_negativo)])
valor_minimo

np.int64(2131)

In [24]:
# Cria um DataFrame df_treino_final concatenando amostras aleatórias de dados_treino_positivo e dados_treino_negativo, com tamanho mínimo igual a valor_minimo, mantendo um estado aleatório consistente
dados_treino_final = pd.concat([dados_treino_positivo.sample(n = valor_minimo, random_state = 69), 
                             dados_treino_negativo.sample(n = valor_minimo, random_state = 69)], 
                            axis = 0, 
                            ignore_index = True)

In [25]:
# Embaralha as linhas do DataFrame dados_treino_final mantendo um estado aleatório consistente e reinicia os índices sem adicionar uma coluna para os índices antigos
dados_treino_final = dados_treino_final.sample(n = len(dados_treino_final), random_state = 69).reset_index(drop = True)

In [26]:
dados_treino_final.shape

(4262, 11)

In [27]:
dados_treino_final['inadimplente'].value_counts()

inadimplente
1    2131
0    2131
Name: count, dtype: int64

In [28]:
# Calcula e imprime a prevalência da classe positiva (inadimplente) no DataFrame df_treino_final, com três casas decimais
print('Balanceamento em Treino(n = %d): %.3f'%(len(dados_treino_final), 
                                               calcula_prevalencia(dados_treino_final['inadimplente'].values)))

Balanceamento em Treino(n = 4262): 0.500


---


# Estudo: Tratamento de Dados Desbalanceados

## Metadados

- **Data:** 2026-05-20
    
- **Tópicos:** #MachineLearning #DataPreparation #ImbalancedData #Sampling
    
- **Linguagem:** Python
    
- **Bibliotecas:** `pandas`, `numpy`, `sklearn`
    

## 📌 1. Configuração e Cenário Inicial

O estudo inicia com a criação de uma massa de dados sintética usando a função `make_classification` para simular um cenário real de análise de risco de crédito (variável alvo: `inadimplente`).

- **Total de registros:** 10.000 linhas
    
- **Características:** 10 variáveis (6 informativas, 2 redundantes e 2 ruídos)
    
- **Prevalência Original:** **30,2%** de inadimplentes (classe positiva) e **69,8%** de clientes adimplentes (classe negativa).
    

## 🔀 2. Divisão Estratégica dos Dados (Holdout)

Antes de qualquer técnica de balanceamento, os dados foram divididos em **Treino (70%)**, **Teste (15%)** e **Validação (15%)**.

> [!IMPORTANT]
> 
> **Ponto Forte do Estudo:** A divisão preservou a prevalência natural das classes (~30% de positivos) em todos os subconjuntos. Isso garante que a validação e o teste reflitam o comportamento real do negócio.

- **Treino:** 7.000 registros (Prevalência: 30,4%)
    
- **Teste:** 1.500 registros (Prevalência: 30,9%)
    
- **Validação:** 1.500 registros (Prevalência: 29,5%)
    

## ⚖️ 3. Aplicação de Random Undersampling (Balanceamento)

Para evitar que o modelo de Machine Learning fique tendencioso a favor da classe majoritária, foi aplicada a técnica de **Random Undersampling** (Subamostragem Aleatória) **apenas no conjunto de treino**.

### Fluxo do Algoritmo Aplicado:

1. **Separação:** O treino foi dividido em `dados_treino_positivo` (2.131 linhas) e `dados_treino_negativo` (4.869 linhas).
    
2. **Cálculo do Limite:** Identificação do volume da menor classe (`valor_minimo` = 2.131).
    
3. **Amostragem e Concatenagem:** Foi extraída uma amostra aleatória de 2.131 registros da classe negativa, juntando-a com os 2.131 registros da classe positiva.
    
4. **Embaralhamento (Shuffle):** Os dados foram misturados para redefinir os índices e retirar qualquer ordenação por classe.
    

### Resultado Final do Treino:

- **Volume:** 4.262 registros.
    
- **Proporção:** **50%** adimplentes / **50%** inadimplentes (Prevalência final: 0,500).
    
---